# 05 · Nonlinearity and interaction insights

The goal: find the curves, thresholds and interactions that feature engineering should
target. Two lenses:

1. **What did the model learn?** SHAP on the points model: the shape of each main
   effect, and pairwise interaction strength.
2. **Where is the market wrong?** Segments where the closing line misses systematically.
   Discovery on 2016–2023; confirmation on 2024–2025, which was never used to find them.

Each insight ends as a feature idea or a pre-registered paper rule.

In [ ]:
import json

import lightgbm as lgb
import numpy as np
import pandas as pd
import shap

from canes_cfb.modeling import FEATURES
from canes_cfb.paths import PROCESSED, RAW, ROOT

features = pd.read_parquet(PROCESSED / "team_games.parquet")
games = pd.read_parquet(RAW / "games.parquet")
tuned = json.loads((ROOT / "models" / "team_points_params.json").read_text())
data = features[features.completed & ~features.shortened & features.season.between(2016, 2023)]
data = data.reset_index(drop=True)

## 1. Model lens: SHAP on the residual points model

Deliberately deeper trees (15 leaves) than the tuned model, so interactions *can* show
up if they exist.

In [ ]:
params = {**tuned["lightgbm"]["params"], "num_leaves": 15, "min_child_samples": 50}
model = lgb.LGBMRegressor(subsample_freq=1, random_state=7, verbose=-1, **params)
model.fit(data[FEATURES], data.points - data.exp_points)
sample = data[data.season >= 2019].sample(2500, random_state=1)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample[FEATURES])
importance = pd.Series(np.abs(shap_values).mean(0), index=FEATURES).sort_values(ascending=False)
importance.head(12).round(3)

### Shapes: mean SHAP by feature decile (points added to the ratings' prediction)

In [ ]:
shapes = {}
for col in importance.index[:8]:
    deciles = pd.qcut(sample[col], 10, duplicates="drop")
    effect = pd.Series(shap_values[:, FEATURES.index(col)], index=sample.index)
    shapes[col] = effect.groupby(deciles, observed=True).mean().round(2).tolist()
pd.DataFrame.from_dict(shapes, orient="index", columns=[f"d{i}" for i in range(1, 11)])

| Feature | Shape | Reading → feature idea |
|---|---|---|
| `exp_total` | +2.2 … −2.2, steepest at the ends | Ratings overstate high-scoring games (regression to the mean) → shrink extremes; **shootout flag** |
| `talent_diff` | −2.0 … flat middle … +2.5 | Only **big** talent gaps matter → tail/threshold feature |
| `ret_ppa` | −1.1 in the lowest decile, flat above the median | Heavy roster turnover hurts → threshold on low returning production |
| `exp_sr` | convex, +1.3 at the top | Elite efficiency carries information that points don't |

### Interactions: SHAP interaction values

In [ ]:
interaction = explainer.shap_interaction_values(sample[FEATURES])
strength = np.abs(interaction).mean(0)
main_total = np.diagonal(strength).sum()
np.fill_diagonal(strength, 0)
pairs = [
    (FEATURES[i], FEATURES[j], 2 * strength[i, j])
    for i in range(len(FEATURES))
    for j in range(i + 1, len(FEATURES))
]
top_pairs = pd.DataFrame(sorted(pairs, key=lambda x: -x[2])[:10], columns=["a", "b", "strength"])
print(f"sum of main effects: {main_total:.2f} points")
top_pairs.round(3)

The strongest interaction (talent gap × scoring environment) is worth ~0.14 points,
against ~10 points of main effects. **The signal is in one-variable curves and
thresholds, not in crosses.** That agrees with the ablations, where an additive model
matched deeper trees.

## 2. Market lens: where does the closing line miss?

Game-level (home row). Over rate vs the closing total; home cover rate vs the closing
spread. Pushes excluded. Quintiles set on 2016–2023.

In [ ]:
rows = features.merge(games[["game_id", "home_id"]], on="game_id")
home = rows[
    (rows.team_id == rows.home_id) & rows.completed & ~rows.shortened & rows.total_close.notna()
].copy()
home["total_result"] = np.sign(home.points + home.points_allowed - home.total_close)
home["cover_result"] = np.sign(home.points - home.points_allowed + home.spread_close)
home["abs_spread"] = home.spread_close.abs()
home["ret_diff"] = home.ret_ppa - home.opp_ret_ppa
candidates = {
    "exp_total": "total_result",
    "total_close": "total_result",
    "game_pace": "total_result",
    "week": "total_result",
    "abs_spread": "cover_result",
    "talent_diff": "cover_result",
    "ret_diff": "cover_result",
    "elo_diff": "cover_result",
    "rest_diff": "cover_result",
}
discover = home[home.season.between(2016, 2023)]
holdout = home[home.season.between(2024, 2025)]


def side_rate(df, result):
    decided = df[result][df[result] != 0]
    return 100 * (decided > 0).mean(), len(decided)


found = []
for col, result in candidates.items():
    edges = pd.qcut(discover[col], 5, retbins=True, duplicates="drop")[1]
    edges[0], edges[-1] = -np.inf, np.inf
    for lo, hi in zip(edges[:-1], edges[1:], strict=True):
        a = discover[(discover[col] > lo) & (discover[col] <= hi)]
        b = holdout[(holdout[col] > lo) & (holdout[col] <= hi)]
        rate_a, n_a = side_rate(a, result)
        rate_b, n_b = side_rate(b, result)
        z = (rate_a / 100 - 0.5) / np.sqrt(0.25 / n_a)
        found.append(
            {
                "feature": col,
                "result": result,
                "bin": f"({lo:.1f}, {hi:.1f}]",
                "2016-23 %": rate_a,
                "n": n_a,
                "z": z,
                "2024-25 %": rate_b,
                "n hold": n_b,
            }
        )
found = pd.DataFrame(found)
found["holds"] = np.sign(found["2016-23 %"] - 50) == np.sign(found["2024-25 %"] - 50)
print(f"{len(found)} segments tested; ~{0.05 * len(found):.0f} expected at |z| >= 2 by chance")
found[found.z.abs() >= 2].sort_values("z", key=abs, ascending=False).round(1)

- **Totals: every surviving segment points the same way.** High expected totals, high
  closing totals and very high pace all go **under**, in discovery *and* in the holdout.
- **Spreads: the "significant" segments flip sign in the holdout** (returning production,
  Elo). That's noise from testing many bins.

## 3. The shootout-under effect, season by season

In [ ]:
decided = home[home.total_result != 0]
threshold = decided.loc[decided.season.between(2016, 2023), "exp_total"].quantile(0.8)
shootout = decided[decided.exp_total > threshold]
stability = pd.DataFrame(
    {
        "shootout games": shootout.groupby("season").size(),
        "shootout under %": shootout.groupby("season").total_result.apply(
            lambda s: 100 * (s < 0).mean()
        ),
        "all games under %": decided.groupby("season").total_result.apply(
            lambda s: 100 * (s < 0).mean()
        ),
    }
)
stability["gap"] = stability["shootout under %"] - stability["all games under %"]
print(f"threshold (top 20% of exp_total, 2016-2023): {threshold:.1f}")
stability.round(1)

In [ ]:
opening = shootout[shootout.total_open.notna()].copy()
opening["vs_open"] = np.sign(opening.points + opening.points_allowed - opening.total_open)
opening = opening[opening.vs_open != 0]
by_season = opening.groupby("season").vs_open.agg(
    games="size", under_pct=lambda s: 100 * (s < 0).mean()
)
pooled = opening[opening.season.between(2021, 2025)]
print(
    f"2021-2025 vs opening total: {100 * (pooled.vs_open < 0).mean():.1f}% under, n={len(pooled)}"
)
by_season.round(1)

**Every season from 2021 to 2025 goes 55–60% under when the ratings expect a shootout**
(2019: 59.9%; 2020, the COVID season: 51.7%; 2015–2018 near 50%), even in over-heavy years (2024: 48% under overall, 55% in shootouts). Against the bettable
opening total it's above break-even in each of 2021–2025 (56.4% pooled, n = 509).
Earlier I read the 2025 market-layer totals result as a pure base-rate artifact. That
was wrong: part of it is this stable effect.

It's the same nonlinearity as the `exp_total` SHAP curve, seen from the market side:
**games that look like shootouts get overestimated by the ratings and by the market.**

**Actions (2026-09-23):**
1. Pre-registered paper rule **B, shootout_under**: under whenever `exp_total > 63.5`,
   tracked separately from rule A in `paper_trading/`.
2. Conflict found on day 1: Ole Miss @ Florida is an *over* for rule A (model 66.8 vs
   59.2) and an *under* for rule B. The points model doesn't shrink shootouts enough.
   → Next feature work: a dedicated **totals model** with these nonlinear features
   (shootout flag, pace tail, talent-gap tails, low returning production) and the
   opening total as the base.